In [ ]:
# Step 1: 
# Set Data Paths + Donor-vs-Donor Classification Configuration

import os

DATA_ROOT = '/mnt/HDD16TB/LanceKam_Lab/Daizong/Project/DLBCL/DLBCL filtered cells-jpg'

# All images are directly in DATA_ROOT (no subfolders)

# Donor-vs-donor task - using donor IDs that start with 'N' and 'R'
# Label 1: donor starting with 'R' (Responder)
# Label 0: donor starting with 'N' (Non-responder)
DONOR_LABEL_MAP = {
    'R': 1,  # All donors starting with 'R' are Responders
    'N': 0,  # All donors starting with 'N' are Non-responders
}

# Split settings
SPLIT_RANDOM_SEED = 15
TRAIN_RATIO = 0.8
VALID_RATIO = 0.1
TEST_RATIO = 0.1

print('Data root set:')
print(f'  Root:  {DATA_ROOT}')
print('  All images are directly in this folder (no class subfolders)')

print('\nDonor classification target (based on first letter):')
print(f'  Donors starting with R -> label 1 (Responder)')
print(f'  Donors starting with N -> label 0 (Non-responder)')

# Verify DATA_ROOT exists
if os.path.exists(DATA_ROOT):
    # Count image files (optional)
    image_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
    image_files = [f for f in os.listdir(DATA_ROOT) 
                   if f.lower().endswith(image_extensions)]
    
    print(f'\n✅ DATA_ROOT exists')
    print(f'   Found {len(image_files)} image files in root directory')
    
    if len(image_files) == 0:
        print('   ⚠️  No image files found in DATA_ROOT!')
else:
    print(f'\n❌ DATA_ROOT NOT FOUND: {DATA_ROOT}')
    print('WARNING: Please verify DATA_ROOT path!')

In [ ]:
# Step 2: Import Libraries
import gc
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from glob import glob
import os
import re
import gc
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.model_selection import train_test_split

print('='*60)
print('SWIN + Metadata - Clean Implementation')
print('='*60)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')

print('\n✅ Imports successful')


In [ ]:
# Step 3: Define Transforms

import torchvision.transforms as T

train_transform = T.Compose([
      T.RandomHorizontalFlip(p=0.5),
      T.RandomRotation(20),
      T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
      T.ToTensor(),
      T.Normalize(mean=[0.5], std=[0.5])
])

val_transform = T.Compose([
      T.ToTensor(),
      T.Normalize(mean=[0.5], std=[0.5])
])

test_transform = T.Compose([
      T.ToTensor(),
      T.Normalize(mean=[0.5], std=[0.5])
])

print('✅ Transform defined')


In [ ]:
# Step 4: Metadata Processing Functions (6-dim, filename suffix)

METADATA_DIM_NAMES = ['cd4', 'cd8', 'naive', 'effector', 'em', 'cm']
CELL_TYPE_INDEX = {'cd4': 0, 'cd8': 1}
SUBTYPE_INDEX = {'naive': 2, 'effector': 3, 'em': 4, 'cm': 5}


def parse_metadata_from_filename(filename):
    """
    Extract metadata from the filename suffix.

    Expected 6-dim order:
        [cd4, cd8, naive, effector, em, cm]

    Examples:
        ..._CD8_Effector.png -> ('cd8', 'effector')
        ..._cd4_naive.jpg -> ('cd4', 'naive')
    """
    stem = Path(filename).stem.lower()
    tokens = [token for token in re.split(r'[_\-\s]+', stem) if token]

    cell_type = next((token for token in reversed(tokens) if token in CELL_TYPE_INDEX), None)
    subtype = next((token for token in reversed(tokens) if token in SUBTYPE_INDEX), None)

    if cell_type is None or subtype is None:
        raise ValueError(
            f'Could not parse metadata from filename: {filename}. '
            'Expected suffix tokens like ..._CD4_Naive or ..._cd8_effector.'
        )

    return cell_type, subtype


def encode_metadata(cell_type, subtype):
    """
    Encode metadata as a 6-dim multi-hot vector:
        [cd4, cd8, naive, effector, em, cm]
    """
    metadata_vector = torch.zeros(len(METADATA_DIM_NAMES), dtype=torch.float32)
    metadata_vector[CELL_TYPE_INDEX[cell_type]] = 1.0
    metadata_vector[SUBTYPE_INDEX[subtype]] = 1.0
    return metadata_vector


test_cases = [
    ('Non-responder_01-03-2026_DLBCL_1to10_109241_sample01_image01_cell01_native_CD4_EM.jpg', 'cd4', 'em'),
    ('H061224F2_20250620_1to10_40min_1_16_cell_11_CD4_EM.jpg', 'cd4', 'em'),
    ('sample_cd8_naive.png', 'cd8', 'naive'),
    ('test_CD4_CM.jpg', 'cd4', 'cm'),
    ('data_cd8_effector.png', 'cd8', 'effector'),
]

print('\nTest cases:')
for filename, expected_ct, expected_st in test_cases:
    ct, st = parse_metadata_from_filename(filename)
    vec = encode_metadata(ct, st)
    print(f'\n  {filename}')
    print(f'    Parsed: ({ct}, {st})')
    print(f'    Expected: ({expected_ct}, {expected_st})')
    print(f'    Encoded: {vec.tolist()}')

print(f'\nMetadata order: {METADATA_DIM_NAMES}')
print('✅ Metadata functions ready (6-dim format)')


In [ ]:
# Step 5: Dataset Class
import os
from pathlib import Path
import cv2
from PIL import Image
import torch
from torch.utils.data import Dataset

class patch_dataset(Dataset):
    """Dataset returning image, label, and 6-dim metadata."""

    def __init__(self, samples_df, image_paths=None, transform=None, split_name='split', data_root=None):
        """
        Args:
            samples_df: DataFrame with columns 'path' and 'label' (and optionally 'filename')
            image_paths: Optional list of image paths (if not provided, uses samples_df['path'])
            transform: Image transforms
            split_name: Name of split (train/val/test)
            data_root: Root directory for images (if paths in samples_df are relative)
        """
        self.transform = transform
        self.split_name = split_name
        
        # Handle data_root
        if data_root is not None:
            self.data_root = Path(data_root)
        else:
            self.data_root = None
        
        # Reset dataframe index
        self.samples_df = samples_df.reset_index(drop=True).copy()
        
        # Determine image paths
        if image_paths is not None:
            self.image_paths = [Path(path) for path in image_paths]
        elif 'path' in self.samples_df.columns:
            # Use paths from dataframe
            if self.data_root:
                self.image_paths = [self.data_root / Path(row['path']) for _, row in self.samples_df.iterrows()]
            else:
                self.image_paths = [Path(row['path']) for _, row in self.samples_df.iterrows()]
        else:
            raise ValueError("Either image_paths must be provided or samples_df must have a 'path' column")
        
        # Print dataset statistics
        if 'label' in self.samples_df.columns:
            label0_n = int((self.samples_df['label'] == 0).sum())
            label1_n = int((self.samples_df['label'] == 1).sum())
            print(f'  Dataset: {split_name}')
            print(f'    Label 0: {label0_n}')
            print(f'    Label 1: {label1_n}')
            print(f'    Total: {len(self.samples_df)}')
        else:
            print(f'  Dataset: {split_name}')
            print(f'    Total: {len(self.samples_df)} (no labels)')
        
        # Preprocess metadata from filenames
        print(f'  Preprocessing metadata for {split_name}...')
        self.metadata_cache = {}
        for idx, img_path in enumerate(self.image_paths):
            try:
                cell_type, subtype = parse_metadata_from_filename(img_path.name)
                self.metadata_cache[idx] = encode_metadata(cell_type, subtype)
            except ValueError as e:
                print(f'    Warning: Could not parse metadata from {img_path.name}: {e}')
                # Use default metadata (CD4 + Naive) as fallback
                self.metadata_cache[idx] = encode_metadata('CD4', 'Naive')
        print('  ✅ Ready')
    
    def __getitem__(self, idx):
        """Return image, metadata, label, and filename"""
        # Get row data
        row = self.samples_df.iloc[idx]
        
        # Get image path
        img_path = self.image_paths[idx]
        
        # Get label if exists
        if 'label' in self.samples_df.columns:
            label = int(row['label'])
        else:
            label = -1  # No label
        
        # Get filename (use basename of image path)
        filename = img_path.name if hasattr(img_path, 'name') else os.path.basename(str(img_path))
        
        # Load image
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f'Failed to read image: {img_path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        
        # Apply transforms
        if self.transform:
            img = self.transform(img)
        
        # Get metadata from cache
        metadata = self.metadata_cache[idx]  # 6-dim vector
        
        # Return as dict or tuple
        return {
            'image': img,
            'metadata': metadata,  # ← This was missing!
            'label': torch.tensor(label, dtype=torch.long) if label != -1 else None,
            'filename': filename
        }
    
    def __len__(self):
        return len(self.samples_df)


print('✅ Dataset class defined')

In [ ]:
# Step 6: Create Datasets and DataLoaders

print('Building donor-vs-donor split from flat image directory...\n')


def extract_donor_prefix(filename):
    """
    Extract donor prefix from filename.
    Looks for patterns like 'N_' or 'R_' at the beginning or in the filename
    """
    # Look for patterns like _N_ or _R_ in the filename
    m = re.search(r'_([NR])_', filename)
    if m:
        return m.group(1)
    
    # Look for patterns at the beginning of the filename
    m = re.match(r'^([NR])', filename)
    if m:
        return m.group(1)
    
    # Look for N or R followed by underscore or dot
    m = re.search(r'([NR])[_.]', filename)
    if m:
        return m.group(1)
    
    return None


def collect_all_images(data_root):
    """Collect all images from flat directory (no class subfolders)."""
    rows = []
    
    # Get all jpg files directly in data_root
    paths = sorted(glob(os.path.join(data_root, '*.jpg')))
    
    print(f"Found {len(paths)} jpg files in {data_root}")
    
    for p in paths:
        fname = os.path.basename(p)
        donor_prefix = extract_donor_prefix(fname)
        
        if donor_prefix is None:
            print(f"⚠️  Warning: Cannot parse donor prefix from filename: {fname}")
            # You can choose to skip or assign a default
            continue
        
        rows.append({
            'path': p,
            'filename': fname,
            'donor_prefix': donor_prefix
        })

    if not rows:
        raise RuntimeError(f'No jpg images found in {data_root} or none could be parsed')

    return pd.DataFrame(rows)


def build_donor_binary_task(df, donor_label_map):
    """Keep only target donors and assign labels by donor prefix."""
    df = df.copy()

    target_prefixes = set(donor_label_map.keys())
    
    print(f"\nTarget donor prefixes: {target_prefixes}")
    print(f"Unique donor prefixes found in data: {sorted(df['donor_prefix'].unique())}")
    print(f"Donor prefixes with counts: {df['donor_prefix'].value_counts().to_dict()}")
    
    task_df = df[df['donor_prefix'].isin(target_prefixes)].copy()

    if task_df.empty:
        print("\n⚠️  WARNING: No samples found for target donor prefixes!")
        print(f"Target prefixes expected: {target_prefixes}")
        print(f"Donor prefixes available: {sorted(df['donor_prefix'].unique())}")
        raise ValueError('No samples found for target donor prefixes. See debug info above.')

    task_df['label'] = task_df['donor_prefix'].map(donor_label_map).astype(int)

    print('\nTarget donor sample counts (before split):')
    print(task_df.groupby(['donor_prefix', 'label']).size().to_dict())

    return task_df


def split_8_1_1(task_df, seed=49):
    """Split all selected samples into train/valid/test = 8:1:1."""
    n_total = len(task_df)
    n_test = int(round(n_total * TEST_RATIO))
    n_valid = int(round(n_total * VALID_RATIO))
    n_train = n_total - n_test - n_valid

    # First split: test
    train_valid_df, test_df = train_test_split(
        task_df,
        test_size=n_test,
        random_state=seed,
        stratify=task_df['label']
    )

    # Second split: valid from remaining
    valid_size_from_remain = n_valid
    train_df, valid_df = train_test_split(
        train_valid_df,
        test_size=valid_size_from_remain,
        random_state=seed,
        stratify=train_valid_df['label']
    )

    train_df = train_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    valid_df = valid_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    assert len(train_df) == n_train, f'Train size mismatch: {len(train_df)} vs {n_train}'
    assert len(valid_df) == n_valid, f'Valid size mismatch: {len(valid_df)} vs {n_valid}'
    assert len(test_df) == n_test, f'Test size mismatch: {len(test_df)} vs {n_test}'

    # Ensure both donor prefixes appear in each split
    target_prefixes = set(DONOR_LABEL_MAP.keys())
    assert target_prefixes.issubset(set(train_df['donor_prefix'])), 'Some target donor prefix missing in train.'
    assert target_prefixes.issubset(set(valid_df['donor_prefix'])), 'Some target donor prefix missing in valid.'
    assert target_prefixes.issubset(set(test_df['donor_prefix'])), 'Some target donor prefix missing in test.'

    # Ensure no duplicated image path across splits
    all_paths = pd.concat([
        train_df[['path']],
        valid_df[['path']],
        test_df[['path']]
    ], ignore_index=True)
    assert all_paths['path'].nunique() == len(all_paths), 'Duplicate image paths found across splits.'

    return train_df, valid_df, test_df


def print_split_report(name, split_df):
    label0_n = int((split_df['label'] == 0).sum())
    label1_n = int((split_df['label'] == 1).sum())

    print(f'{name}:')
    print(f'  Samples: {len(split_df)}')
    print(f'  Label 0 (N-prefix): {label0_n}')
    print(f'  Label 1 (R-prefix): {label1_n}')
    print(f'  Donor breakdown: {split_df.groupby(["donor_prefix", "label"]).size().to_dict()}')


# 1) Collect all images from flat directory
all_images_df = collect_all_images(DATA_ROOT)
print(f'\nCollected images from {DATA_ROOT}: {len(all_images_df)}')
print("\nFirst 5 filenames with their donor prefixes:")
for i, row in all_images_df.head(5).iterrows():
    print(f"  {i+1}. {row['filename']} -> donor prefix: {row['donor_prefix']}")

# 2) Build donor-vs-donor binary task (based on N/R prefix)
task_df = build_donor_binary_task(all_images_df, DONOR_LABEL_MAP)

print('\nTask label mapping:')
print(f'  Donors starting with R -> label 1 (Responder)')
print(f'  Donors starting with N -> label 0 (Non-responder)')

# 3) Split with 8:1:1
train_df, valid_df, test_df = split_8_1_1(task_df, seed=SPLIT_RANDOM_SEED)

print('\nFinal split report:')
print_split_report('Train', train_df)
print_split_report('Valid', valid_df)
print_split_report('Test', test_df)

n_total = len(task_df)
print('\nGlobal ratio check (count and fraction):')
print(f'  Train: {len(train_df)} ({len(train_df)/n_total:.4f})')
print(f'  Valid: {len(valid_df)} ({len(valid_df)/n_total:.4f})')
print(f'  Test:  {len(test_df)} ({len(test_df)/n_total:.4f})')

# 4) Create datasets and dataloaders
print('\nCreating datasets...\n')
train_dataset = patch_dataset(train_df, transform=train_transform, split_name='train')
print()
valid_dataset = patch_dataset(valid_df, transform=val_transform, split_name='valid')
print()
test_dataset = patch_dataset(test_df, transform=test_transform, split_name='test')

print('\nDataset summary:')
print(f'  Train: {len(train_dataset)}')
print(f'  Valid: {len(valid_dataset)}')
print(f'  Test:  {len(test_dataset)}')

BATCH_SIZE = 64

trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
testloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'\nDataLoaders (batch_size={BATCH_SIZE}):')
print(f'  Train batches: {len(trainloader)}')
print(f'  Valid batches: {len(validloader)}')
print(f'  Test batches:  {len(testloader)}')

sample = next(iter(trainloader))
print(f'\nVerification:')
print(f'  Image shape: {sample["image"].shape}')
print(f'  Labels dtype: {sample["label"].dtype}')
print(f'  Metadata shape: {sample["metadata"].shape}')
print(f'  Sample filename: {sample["filename"][0]}')
print(f'  Sample metadata: {sample["metadata"][0].tolist()}')

print('\n✅ Donor-vs-donor split + dataloaders ready')

## Step 7: Baseline Model (No FiLM, No Metadata)

Standard SWIN + MLP classifier. No metadata, no FiLM modulation.
This is trained first and its best weights are used as the frozen backbone for FiLM fine-tuning.

In [ ]:
# Step 7: Baseline Model Class - SWIN only, no FiLM, no metadata

import torch
import torch.nn as nn

class SWINBaseline(nn.Module):
    """
    Pure SWIN classifier with no metadata and no FiLM modulation.

    Architecture:
        SWIN backbone (768-dim output)
        -> head: 768 -> 128 -> 64 -> 1

    This model is trained first as the baseline.
    Its best checkpoint is then used as the frozen backbone
    for the FiLM fine-tuning stage.
    """

    def __init__(self, swin_model, dropout_rate=0.5):
        super().__init__()

        # Replace SWIN's default head with our custom MLP classifier
        n_inputs = swin_model.head.in_features  # 768 for swin_tiny
        swin_model.head = nn.Sequential(
            nn.Linear(n_inputs, 128),
            nn.BatchNorm1d(128),
            nn.PReLU(num_parameters=1),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.PReLU(num_parameters=1),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        self.model = swin_model

    def forward(self, x):
        # Image only - no metadata passed
        return self.model(x)

print('✅ SWINBaseline class defined')
print('   Architecture: SWIN backbone -> head(768->128->64->1)')
print('   No metadata, no FiLM')


## Step 8: Train Baseline Model

Train the no-FiLM baseline from scratch. Save the best checkpoint based on validation loss.

In [ ]:
# Step 8: Create and Train Baseline Model

import gc
import numpy as np
from torch.optim import Adam, lr_scheduler

# Checkpoint path for baseline best model
BASELINE_CKPT = (
    '/mnt/HDD16TB/LanceKam_Lab/Daizong/Project/DLBCL/DLBCL/Best model/'
    'best_model_baseline_no_film.pth'
)

# ── Clean up memory ───────────────────────────────────────────────
if 'model' in globals(): del model
if 'baseline_model' in globals(): del baseline_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Build baseline model ──────────────────────────────────────────
print('Loading SWIN backbone...')
base_swin = torch.hub.load(
    'SharanSMenon/swin-transformer-hub:main',
    'swin_tiny_patch4_window7_224', pretrained=True
)
baseline_model = SWINBaseline(base_swin, dropout_rate=0.5).to(device)
del base_swin
gc.collect()

total_params = sum(p.numel() for p in baseline_model.parameters())
print(f'Total parameters: {total_params:,} ({total_params/1e6:.2f}M)')
print('✅ Baseline model ready')

# ── Training config ───────────────────────────────────────────────
BASELINE_LR     = 1e-5
BASELINE_EPOCHS = 100

optimizer_bl = Adam(baseline_model.parameters(), lr=BASELINE_LR)
scheduler_bl = lr_scheduler.ReduceLROnPlateau(
    optimizer_bl, mode='min', factor=0.5, patience=10,
    verbose=True, threshold=0.0001, threshold_mode='rel',
    cooldown=5, min_lr=1e-8
)
loss_fn_bl = nn.BCELoss().to(device)

# ── Training loop ─────────────────────────────────────────────────
print('\n' + '='*60)
print('BASELINE TRAINING START')
print('='*60)

best_bl_loss = float('inf')
best_bl_acc  = 0.0
bl_train_loss_hist = []; bl_train_acc_hist = []
bl_valid_loss_hist = []; bl_valid_acc_hist = []
bl_lr_hist = []

for epoch in range(BASELINE_EPOCHS):
    print(f'\nEpoch {epoch+1}/{BASELINE_EPOCHS}')
    print('-'*50)

    # Train
    baseline_model.train()
    tr_losses, tr_accs = [], []
    for batch in trainloader:
        imgs = batch['image'].to(device)
        lbls = batch['label'].float().view(-1, 1).to(device)
        out  = baseline_model(imgs)          # no metadata
        loss = loss_fn_bl(out, lbls)
        preds = (out > 0.5).float()
        acc   = (preds == lbls).float().mean()
        optimizer_bl.zero_grad()
        loss.backward()
        optimizer_bl.step()
        tr_losses.append(loss.item())
        tr_accs.append(acc.item())

    # Validate
    baseline_model.eval()
    vl_losses, vl_accs = [], []
    with torch.no_grad():
        for batch in validloader:
            imgs = batch['image'].to(device)
            lbls = batch['label'].float().view(-1, 1).to(device)
            out  = baseline_model(imgs)
            loss = loss_fn_bl(out, lbls)
            preds = (out > 0.5).float()
            vl_losses.append(loss.item())
            vl_accs.append((preds == lbls).float().mean().item())

    ep_tr_loss = np.mean(tr_losses); ep_tr_acc = np.mean(tr_accs)
    ep_vl_loss = np.mean(vl_losses); ep_vl_acc = np.mean(vl_accs)
    bl_train_loss_hist.append(ep_tr_loss); bl_train_acc_hist.append(ep_tr_acc)
    bl_valid_loss_hist.append(ep_vl_loss); bl_valid_acc_hist.append(ep_vl_acc)

    cur_lr = optimizer_bl.param_groups[0]['lr']
    bl_lr_hist.append(cur_lr)
    scheduler_bl.step(ep_vl_loss)

    print(f'Train: Loss={ep_tr_loss:.4f}, Acc={ep_tr_acc:.4f}')
    print(f'Valid: Loss={ep_vl_loss:.4f}, Acc={ep_vl_acc:.4f}')
    print(f'LR:    {cur_lr:.2e}')

    if ep_vl_loss < best_bl_loss:
        best_bl_loss = ep_vl_loss
        best_bl_acc  = ep_vl_acc
        torch.save(baseline_model.state_dict(), BASELINE_CKPT)
        print(f'  ✅ Best baseline saved (Val Loss: {best_bl_loss:.4f}, Acc: {best_bl_acc:.4f})')

print('\n' + '='*60)
print('BASELINE TRAINING COMPLETE')
print(f'Best Val Loss: {best_bl_loss:.4f}  |  Best Val Acc: {best_bl_acc:.4f}')
print('='*60)


## Step 8b: Baseline Training Curves

Plot loss, accuracy, and learning rate curves for the baseline model.

In [ ]:
# Step 8b: Plot Baseline Training Curves

import matplotlib.pyplot as plt
import numpy as np

def plot_baseline_training_curves(train_loss, train_acc, valid_loss, valid_acc, lr_hist):
    """Plot and save baseline training curves as separate figures."""

    epochs = range(1, len(train_loss) + 1)

    # ── Figure 1: Loss curve ──────────────────────────────────────
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, 'b-', linewidth=2, label='Train Loss')
    plt.plot(epochs, valid_loss, 'r-', linewidth=2, label='Valid Loss')
    best_epoch = int(np.argmin(valid_loss)) + 1
    plt.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.6,
                label=f'Best epoch {best_epoch}')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Baseline: Training and Validation Loss', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('baseline_loss_curve.png', dpi=300)
    plt.show()
    print('✅ Saved: baseline_loss_curve.png')

    # ── Figure 2: Accuracy curve ──────────────────────────────────
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_acc, 'b-', linewidth=2, label='Train Accuracy')
    plt.plot(epochs, valid_acc, 'r-', linewidth=2, label='Valid Accuracy')
    best_acc_epoch = int(np.argmax(valid_acc)) + 1
    plt.axvline(x=best_acc_epoch, color='g', linestyle='--', alpha=0.6,
                label=f'Best epoch {best_acc_epoch}')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Baseline: Training and Validation Accuracy', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('baseline_accuracy_curve.png', dpi=300)
    plt.show()
    print('✅ Saved: baseline_accuracy_curve.png')

    # ── Figure 3: Learning rate curve ────────────────────────────
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, lr_hist, 'g-', linewidth=2, marker='o', markersize=3)
    # Mark LR drop points
    for i in range(1, len(lr_hist)):
        if lr_hist[i] < lr_hist[i - 1]:
            plt.axvline(x=i + 1, color='r', linestyle='--', alpha=0.4)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Learning Rate', fontsize=12)
    plt.title('Baseline: Learning Rate Schedule', fontsize=14, fontweight='bold')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('baseline_lr_curve.png', dpi=300)
    plt.show()
    print('✅ Saved: baseline_lr_curve.png')

    # ── Print statistics ──────────────────────────────────────────
    print('\n' + '='*60)
    print('BASELINE TRAINING STATISTICS')
    print('='*60)
    print(f'Total epochs:           {len(train_loss)}')
    print(f'Best valid loss:        {np.min(valid_loss):.4f} at epoch {int(np.argmin(valid_loss))+1}')
    print(f'Best valid accuracy:    {np.max(valid_acc):.4f} ({np.max(valid_acc)*100:.2f}%) at epoch {int(np.argmax(valid_acc))+1}')
    print(f'Final train loss:       {train_loss[-1]:.4f}')
    print(f'Final valid loss:       {valid_loss[-1]:.4f}')
    print(f'Final train accuracy:   {train_acc[-1]:.4f} ({train_acc[-1]*100:.2f}%)')
    print(f'Final valid accuracy:   {valid_acc[-1]:.4f} ({valid_acc[-1]*100:.2f}%)')
    print(f'LR range:               {np.min(lr_hist):.2e} to {np.max(lr_hist):.2e}')

plot_baseline_training_curves(
    bl_train_loss_hist,
    bl_train_acc_hist,
    bl_valid_loss_hist,
    bl_valid_acc_hist,
    bl_lr_hist
)


## Step 9: Evaluate Baseline on Test Set

In [ ]:
# Step 9: Evaluate Baseline on Test Set

import numpy as np
from sklearn.metrics import roc_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model_image_only(model, loader, device, tag='TEST'):
    """Evaluate a baseline model that takes only images (no metadata)."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(device)
            lbls  = batch['label'].float().view(-1, 1).to(device)
            out   = model(imgs)
            preds = (out > 0.5).float()
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(lbls.cpu().numpy().flatten())
            all_probs.extend(out.cpu().numpy().flatten())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc = (all_preds == all_labels).mean()
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    auc_val = auc(fpr, tpr)
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()

    print('\n' + '='*60)
    print(f'BASELINE EVALUATION ({tag})')
    print('='*60)
    print(f'Accuracy:    {acc:.4f} ({acc*100:.2f}%)')
    print(f'AUC:         {auc_val:.4f}')
    print(f'Sensitivity: {tp/(tp+fn):.4f}')
    print(f'Specificity: {tn/(tn+fp):.4f}')
    print(f'Confusion Matrix:')
    print(f'              Predicted')
    print(f'                     Non-responder  Responder')
    print(f'Actual  Non-responder  {cm[0,0]:4d}  {cm[0,1]:4d}')
    print(f'        Responder      {cm[1,0]:4d}  {cm[1,1]:4d}')

    return acc, auc_val, fpr, tpr

# Load best baseline checkpoint
print(f'Loading best baseline checkpoint: {BASELINE_CKPT}')
baseline_model.load_state_dict(torch.load(BASELINE_CKPT, map_location=device))

bl_valid_acc, bl_valid_auc, _, _ = evaluate_model_image_only(
    baseline_model, validloader, device, tag='VALID'
)
bl_test_acc, bl_test_auc, bl_fpr, bl_tpr = evaluate_model_image_only(
    baseline_model, testloader, device, tag='TEST'
)


## Step 10: FiLM Model Class

Build `SWINWithMetadata` with the following freeze strategy:

| Component | Status | Reason |
|---|---|---|
| SWIN backbone | **Frozen** | Loaded from baseline checkpoint |
| image_projector (768→128) | **Frozen** | Weights copied from baseline head, already well-trained |
| tabnet_encoder | **Trainable** | Randomly initialized, learns FiLM params from metadata |
| classifier (128→64→1) | **Trainable** | Randomly initialized, re-learns from FiLM-modulated features |

Key fix: `gamma = 1.0 + gamma_raw` so gamma starts at 1 (identity), guaranteeing FiLM cannot compress image features at initialization.

In [ ]:
# Step 10: FiLM Model Class with frozen baseline backbone
# Key design decisions:
#   1. SWIN backbone weights loaded from baseline checkpoint -> FROZEN
#   2. image_projector (768->128) weights transferred from baseline head -> FROZEN
#   3. classifier (128->64->1) randomly initialized -> TRAINABLE
#   4. tabnet_encoder randomly initialized -> TRAINABLE
#   5. gamma = 1.0 + gamma_raw  (identity at initialization)

import torch
import torch.nn as nn
import torch.nn.functional as F


class TabNetBlock(nn.Module):
    """Simplified TabNet-style attention block for metadata processing."""

    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.3):
        super().__init__()
        self.feature_transformer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.PReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.PReLU(),
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.PReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
        self.output_projection = nn.Linear(hidden_dim, output_dim)
        self.feature_mask = None

    def forward(self, x, prior_scale=None):
        residual    = x
        transformed = self.feature_transformer(x)
        attn_mask   = self.attention(transformed)
        if prior_scale is not None:
            attn_mask = attn_mask * prior_scale
        self.feature_mask  = attn_mask
        masked_features    = x * attn_mask
        updated_features   = masked_features + residual
        output             = self.output_projection(transformed)
        return output, attn_mask, updated_features


class TabNetEncoder(nn.Module):
    """TabNet encoder: metadata (6-dim) -> 256-dim (gamma_raw + beta)."""

    def __init__(self, input_dim=6, hidden_dim=32, output_dim=128, n_steps=3, dropout_rate=0.3):
        super().__init__()
        self.n_steps    = n_steps
        self.output_dim = output_dim
        self.initial_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.PReLU(),
            nn.Dropout(dropout_rate),
        )
        self.attention_blocks = nn.ModuleList([
            TabNetBlock(hidden_dim, hidden_dim, output_dim, dropout_rate)
            for _ in range(n_steps)
        ])
        # Output dim*2: first half = gamma_raw, second half = beta
        self.final_projection = nn.Sequential(
            nn.Linear(output_dim * n_steps, output_dim * 2),
            nn.BatchNorm1d(output_dim * 2),
            nn.PReLU(),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        x            = self.initial_transform(x)
        step_outputs = []
        prior_scale  = None
        for block in self.attention_blocks:
            step_out, attn_mask, x = block(x, prior_scale)
            step_outputs.append(step_out)
            prior_scale = attn_mask if prior_scale is None else prior_scale * (1 - attn_mask)
        aggregated = torch.cat(step_outputs, dim=1)
        return self.final_projection(aggregated)  # [batch, output_dim*2]

    def get_attention_masks(self, x):
        x = self.initial_transform(x)
        masks       = []
        prior_scale = None
        for block in self.attention_blocks:
            _, attn_mask, x = block(x, prior_scale)
            masks.append(attn_mask)
            prior_scale = attn_mask if prior_scale is None else prior_scale * (1 - attn_mask)
        return masks


class SWINWithMetadata(nn.Module):
    """
    FiLM model built on top of frozen baseline backbone + frozen image_projector.

    Frozen (from baseline checkpoint):
        SWIN backbone:     patch_embed, pos_drop, layers, norm, avgpool
        image_projector:   768 -> 128  (weights copied from baseline head[0:4])

    Trainable (randomly initialized):
        tabnet_encoder:    metadata(6) -> 256 (gamma_raw + beta)
        classifier:        128 -> 64 -> 1

    FiLM modulation:
        gamma = 1.0 + gamma_raw   <- identity fix, gamma starts at 1
        output = gamma * x + beta

    Rationale for freezing image_projector:
        The 768->128 projection was already learned by the baseline.
        It produces a well-trained 128-dim image representation.
        FiLM is injected at this 128-dim space.
        Only the classifier (128->64->1) needs to re-learn how to interpret
        FiLM-modulated features, since the modulation changes the distribution.
    """

    def __init__(self, baseline_checkpoint_path, device, metadata_dim=6,
                 tabnet_hidden_dim=32, tabnet_steps=3, dropout_rate=0.5):
        super().__init__()

        # ── Load baseline checkpoint ──────────────────────────────
        print('Loading baseline checkpoint...')
        tmp_swin = torch.hub.load(
            'SharanSMenon/swin-transformer-hub:main',
            'swin_tiny_patch4_window7_224', pretrained=False
        )
        tmp_baseline = SWINBaseline(tmp_swin, dropout_rate=dropout_rate)
        ckpt = torch.load(baseline_checkpoint_path, map_location='cpu')
        tmp_baseline.load_state_dict(ckpt, strict=True)
        print('✅ Baseline weights loaded')

        # ── Extract and freeze SWIN backbone ─────────────────────
        inner            = tmp_baseline.model
        self.patch_embed = inner.patch_embed
        self.pos_drop    = inner.pos_drop
        self.layers      = inner.layers
        self.norm        = inner.norm
        self.avgpool     = inner.avgpool

        for p in self.patch_embed.parameters(): p.requires_grad = False
        for p in self.pos_drop.parameters():    p.requires_grad = False
        for p in self.layers.parameters():      p.requires_grad = False
        for p in self.norm.parameters():        p.requires_grad = False
        print('✅ SWIN backbone frozen')

        # ── Build image_projector and transfer weights from baseline head ──
        # Baseline head structure:
        #   head[0] = Linear(768, 128)
        #   head[1] = BatchNorm1d(128)
        #   head[2] = PReLU
        #   head[3] = Dropout
        #   head[4] = Linear(128, 64)   <- NOT copied (classifier part)
        #   ...
        self.first_mid_dim  = 128
        self.second_mid_dim = 64

        self.image_projector = nn.Sequential(
            nn.Linear(768, self.first_mid_dim),
            nn.BatchNorm1d(self.first_mid_dim),
            nn.PReLU(num_parameters=1),
            nn.Dropout(dropout_rate),
        )

        # Copy 768->128 weights from baseline head into image_projector
        baseline_head = inner.head
        with torch.no_grad():
            # Linear(768->128): weight and bias
            self.image_projector[0].weight.copy_(baseline_head[0].weight)
            self.image_projector[0].bias.copy_(baseline_head[0].bias)
            # BatchNorm1d(128): weight, bias, running_mean, running_var
            self.image_projector[1].weight.copy_(baseline_head[1].weight)
            self.image_projector[1].bias.copy_(baseline_head[1].bias)
            self.image_projector[1].running_mean.copy_(baseline_head[1].running_mean)
            self.image_projector[1].running_var.copy_(baseline_head[1].running_var)
            # PReLU: weight
            self.image_projector[2].weight.copy_(baseline_head[2].weight)
        print('✅ image_projector weights transferred from baseline head (768->128)')

        # Freeze image_projector (weights already learned by baseline)
        for p in self.image_projector.parameters():
            p.requires_grad = False
        print('✅ image_projector frozen')

        del tmp_baseline, tmp_swin

        # ── Trainable: tabnet_encoder ─────────────────────────────
        # Randomly initialized, learns gamma_raw and beta from metadata
        self.tabnet_encoder = TabNetEncoder(
            input_dim=metadata_dim,
            hidden_dim=tabnet_hidden_dim,
            output_dim=self.first_mid_dim,
            n_steps=tabnet_steps,
            dropout_rate=dropout_rate
        )

        # ── Trainable: classifier (128->64->1) ────────────────────
        # Randomly initialized. Needs to re-learn from FiLM-modulated features.
        # Cannot reuse baseline head[4:] because FiLM changes the input distribution.
        self.classifier = nn.Sequential(
            nn.Linear(self.first_mid_dim, self.second_mid_dim),
            nn.BatchNorm1d(self.second_mid_dim),
            nn.PReLU(num_parameters=1),
            nn.Dropout(dropout_rate),
            nn.Linear(self.second_mid_dim, 1),
            nn.Sigmoid(),
        )

        self._init_trainable_weights()

    def _init_trainable_weights(self):
        """Initialize only the trainable components (tabnet_encoder + classifier)."""
        for m in self.tabnet_encoder.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

        # Set final projection bias: gamma_raw=0 at init -> gamma = 1.0 + 0 = 1
        last = self.tabnet_encoder.final_projection[0]
        if isinstance(last, nn.Linear):
            nn.init.normal_(last.weight, std=0.01)
            nn.init.zeros_(last.bias)

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x, metadata):
        if metadata.device != x.device:
            metadata = metadata.to(x.device)

        # Frozen SWIN backbone
        x = self.patch_embed(x)
        x = self.pos_drop(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = self.avgpool(x.transpose(1, 2))
        x = torch.flatten(x, 1)           # [batch, 768]

        # Frozen image projection (768->128)
        x = self.image_projector(x)        # [batch, 128]

        # Trainable FiLM parameter generation from metadata
        film_params     = self.tabnet_encoder(metadata)     # [batch, 256]
        gamma_raw, beta = film_params.chunk(2, dim=1)       # Each [batch, 128]

        # gamma = 1 + gamma_raw: identity at init, learns offsets
        gamma = 1.0 + gamma_raw

        # FiLM modulation
        x = gamma * x + beta               # [batch, 128]

        # Trainable classifier (128->64->1)
        return self.classifier(x)

    def get_film_params(self, metadata):
        """Return gamma (after +1 fix) and beta."""
        film_params     = self.tabnet_encoder(metadata)
        gamma_raw, beta = film_params.chunk(2, dim=1)
        gamma           = 1.0 + gamma_raw
        return gamma, beta

    def get_attention_masks(self, metadata):
        return self.tabnet_encoder.get_attention_masks(metadata)


print('✅ SWINWithMetadata class defined')
print('   Frozen:    SWIN backbone + image_projector (768->128) from baseline')
print('   Trainable: tabnet_encoder + classifier (128->64->1)')
print('   gamma = 1.0 + gamma_raw  (identity fix)')


## Step 11: Instantiate FiLM Model

In [ ]:
# Step 11: Instantiate FiLM model

import gc

FILM_CKPT = (
    '/mnt/HDD16TB/LanceKam_Lab/Daizong/Project/DLBCL/DLBCL/Best model/'
    'best_model_FiLM_frozen_backbone.pth'
)

# Clean up memory
if 'film_model' in globals(): del film_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Build FiLM model
# Inside __init__:
#   - SWIN backbone loaded from baseline + frozen
#   - image_projector weights transferred from baseline head[0:4] + frozen
#   - tabnet_encoder randomly initialized (trainable)
#   - classifier randomly initialized (trainable)
film_model = SWINWithMetadata(
    baseline_checkpoint_path=BASELINE_CKPT,
    device=device,
    metadata_dim=6,
    tabnet_hidden_dim=32,
    tabnet_steps=3,
    dropout_rate=0.5
).to(device)

# ── Parameter summary ─────────────────────────────────────────────
total_p     = sum(p.numel() for p in film_model.parameters())
trainable_p = sum(p.numel() for p in film_model.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print(f'\nParameter summary:')
print(f'  Total:     {total_p:,} ({total_p/1e6:.2f}M)')
print(f'  Trainable: {trainable_p:,}  <- tabnet_encoder + classifier')
print(f'  Frozen:    {frozen_p:,}  <- SWIN backbone + image_projector')

# Verify which components are frozen / trainable
print('\nFreeze status per component:')
components = {
    'patch_embed':     film_model.patch_embed,
    'pos_drop':        film_model.pos_drop,
    'layers':          film_model.layers,
    'norm':            film_model.norm,
    'image_projector': film_model.image_projector,
    'tabnet_encoder':  film_model.tabnet_encoder,
    'classifier':      film_model.classifier,
}
for name, module in components.items():
    params = list(module.parameters())
    if not params:
        print(f'  {name:20s}: no parameters')
    else:
        is_frozen = not any(p.requires_grad for p in params)
        status    = 'FROZEN' if is_frozen else 'TRAINABLE'
        n_params  = sum(p.numel() for p in params)
        print(f'  {name:20s}: {status}  ({n_params:,} params)')

# Verify forward pass and initial gamma
test_img  = torch.randn(2, 3, 224, 224).to(device)
test_meta = torch.randn(2, 6).to(device)
with torch.no_grad():
    test_out = film_model(test_img, test_meta)
    g, b     = film_model.get_film_params(test_meta)
print(f'\nForward pass: input {test_img.shape} -> output {test_out.shape}')
print(f'Initial gamma mean: {g.mean().item():.4f}  (should be close to 1.0)')
print(f'Initial beta  mean: {b.mean().item():.4f}  (should be close to 0.0)')
print('\n✅ FiLM model ready')


## Step 12: Train FiLM Model (Frozen Backbone)

Only `image_projector`, `tabnet_encoder`, and `classifier` are trained. The SWIN backbone remains frozen (same weights as the best baseline).

In [ ]:
# Step 12: Train FiLM Model with frozen backbone

from torch.optim import Adam, lr_scheduler

FILM_LR     = 1e-5   # Higher LR than baseline since only new components trained
FILM_EPOCHS = 100

# Only optimize trainable parameters (backbone is frozen)
trainable_params = [p for p in film_model.parameters() if p.requires_grad]
opt_film = Adam(trainable_params, lr=FILM_LR)
sch_film = lr_scheduler.ReduceLROnPlateau(
    opt_film, mode='min', factor=0.5, patience=10,
    verbose=True, threshold=0.0001, threshold_mode='rel',
    cooldown=5, min_lr=1e-8
)
loss_fn_film = nn.BCELoss().to(device)


def print_film_stats(model, tag=''):
    """Print gamma and beta statistics for a sample batch."""
    model.eval()
    with torch.no_grad():
        sample_meta = next(iter(validloader))['metadata'].to(device)
        g, b = model.get_film_params(sample_meta)
    g, b = g.cpu().numpy(), b.cpu().numpy()
    print(f'  FiLM {tag}:')
    print(f'    gamma — mean={g.mean():.4f}, std={g.std():.4f}, min={g.min():.4f}, max={g.max():.4f}')
    print(f'    beta  — mean={b.mean():.4f}, std={b.std():.4f}, min={b.min():.4f}, max={b.max():.4f}')
    model.train()


print('Initial FiLM params (before training):')
print_film_stats(film_model, tag='[init]')

print('\n' + '='*60)
print('FiLM TRAINING START (frozen backbone)')
print('='*60)

best_film_loss = float('inf')
best_film_acc  = 0.0
film_train_loss_hist = []; film_train_acc_hist = []
film_valid_loss_hist = []; film_valid_acc_hist = []
film_lr_hist = []

for epoch in range(FILM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{FILM_EPOCHS}')
    print('-'*50)

    # Train
    film_model.train()
    tr_losses, tr_accs = [], []
    for batch in trainloader:
        imgs = batch['image'].to(device)
        meta = batch['metadata'].to(device)
        lbls = batch['label'].float().view(-1, 1).to(device)
        out  = film_model(imgs, meta)
        loss = loss_fn_film(out, lbls)
        preds = (out > 0.5).float()
        acc   = (preds == lbls).float().mean()
        opt_film.zero_grad()
        loss.backward()
        opt_film.step()
        tr_losses.append(loss.item())
        tr_accs.append(acc.item())

    # Validate
    film_model.eval()
    vl_losses, vl_accs = [], []
    with torch.no_grad():
        for batch in validloader:
            imgs = batch['image'].to(device)
            meta = batch['metadata'].to(device)
            lbls = batch['label'].float().view(-1, 1).to(device)
            out  = film_model(imgs, meta)
            loss = loss_fn_film(out, lbls)
            preds = (out > 0.5).float()
            vl_losses.append(loss.item())
            vl_accs.append((preds == lbls).float().mean().item())

    ep_tr_loss = np.mean(tr_losses); ep_tr_acc = np.mean(tr_accs)
    ep_vl_loss = np.mean(vl_losses); ep_vl_acc = np.mean(vl_accs)
    film_train_loss_hist.append(ep_tr_loss); film_train_acc_hist.append(ep_tr_acc)
    film_valid_loss_hist.append(ep_vl_loss); film_valid_acc_hist.append(ep_vl_acc)

    cur_lr = opt_film.param_groups[0]['lr']
    film_lr_hist.append(cur_lr)
    sch_film.step(ep_vl_loss)

    print(f'Train: Loss={ep_tr_loss:.4f}, Acc={ep_tr_acc:.4f}')
    print(f'Valid: Loss={ep_vl_loss:.4f}, Acc={ep_vl_acc:.4f}')
    print(f'LR:    {cur_lr:.2e}')
    print_film_stats(film_model, tag=f'epoch {epoch+1}')

    if ep_vl_loss < best_film_loss:
        best_film_loss = ep_vl_loss
        best_film_acc  = ep_vl_acc
        torch.save(film_model.state_dict(), FILM_CKPT)
        print(f'  ✅ Best FiLM model saved (Val Loss: {best_film_loss:.4f}, Acc: {best_film_acc:.4f})')

print('\n' + '='*60)
print('FiLM TRAINING COMPLETE')
print(f'Best Val Loss: {best_film_loss:.4f}  |  Best Val Acc: {best_film_acc:.4f}')
print('='*60)


## Step 12b: FiLM Training Curves

Plot loss, accuracy, and learning rate curves for the FiLM model.

In [ ]:
# Step 12b: Plot FiLM Training Curves

import matplotlib.pyplot as plt
import numpy as np

def plot_film_training_curves(train_loss, train_acc, valid_loss, valid_acc, lr_hist):
    """Plot and save FiLM model training curves as separate figures."""

    epochs = range(1, len(train_loss) + 1)

    # ── Figure 1: Loss curve ──────────────────────────────────────
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, 'b-', linewidth=2, label='Train Loss')
    plt.plot(epochs, valid_loss, 'r-', linewidth=2, label='Valid Loss')
    best_epoch = int(np.argmin(valid_loss)) + 1
    plt.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.6,
                label=f'Best epoch {best_epoch}')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('FiLM: Training and Validation Loss', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('film_loss_curve.png', dpi=300)
    plt.show()
    print('✅ Saved: film_loss_curve.png')

    # ── Figure 2: Accuracy curve ──────────────────────────────────
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_acc, 'b-', linewidth=2, label='Train Accuracy')
    plt.plot(epochs, valid_acc, 'r-', linewidth=2, label='Valid Accuracy')
    best_acc_epoch = int(np.argmax(valid_acc)) + 1
    plt.axvline(x=best_acc_epoch, color='g', linestyle='--', alpha=0.6,
                label=f'Best epoch {best_acc_epoch}')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('FiLM: Training and Validation Accuracy', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('film_accuracy_curve.png', dpi=300)
    plt.show()
    print('✅ Saved: film_accuracy_curve.png')

    # ── Figure 3: Learning rate curve ────────────────────────────
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, lr_hist, 'g-', linewidth=2, marker='o', markersize=3)
    for i in range(1, len(lr_hist)):
        if lr_hist[i] < lr_hist[i - 1]:
            plt.axvline(x=i + 1, color='r', linestyle='--', alpha=0.4)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Learning Rate', fontsize=12)
    plt.title('FiLM: Learning Rate Schedule', fontsize=14, fontweight='bold')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('film_lr_curve.png', dpi=300)
    plt.show()
    print('✅ Saved: film_lr_curve.png')

    # ── Print statistics ──────────────────────────────────────────
    print('\n' + '='*60)
    print('FiLM TRAINING STATISTICS')
    print('='*60)
    print(f'Total epochs:           {len(train_loss)}')
    print(f'Best valid loss:        {np.min(valid_loss):.4f} at epoch {int(np.argmin(valid_loss))+1}')
    print(f'Best valid accuracy:    {np.max(valid_acc):.4f} ({np.max(valid_acc)*100:.2f}%) at epoch {int(np.argmax(valid_acc))+1}')
    print(f'Final train loss:       {train_loss[-1]:.4f}')
    print(f'Final valid loss:       {valid_loss[-1]:.4f}')
    print(f'Final train accuracy:   {train_acc[-1]:.4f} ({train_acc[-1]*100:.2f}%)')
    print(f'Final valid accuracy:   {valid_acc[-1]:.4f} ({valid_acc[-1]*100:.2f}%)')
    print(f'LR range:               {np.min(lr_hist):.2e} to {np.max(lr_hist):.2e}')

plot_film_training_curves(
    film_train_loss_hist,
    film_train_acc_hist,
    film_valid_loss_hist,
    film_valid_acc_hist,
    film_lr_hist
)


## Step 13: Evaluate FiLM Model and Final Comparison

In [ ]:
# Step 13: Evaluate FiLM model on test set and compare with baseline

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import roc_curve, auc, confusion_matrix

def evaluate_film_model(model, loader, device, tag='TEST'):
    """Evaluate FiLM model that takes image + metadata."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    gamma_batches, beta_batches = [], []

    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(device)
            meta  = batch['metadata'].to(device)
            lbls  = batch['label'].float().view(-1, 1).to(device)
            out   = model(imgs, meta)
            preds = (out > 0.5).float()
            g, b  = model.get_film_params(meta)
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(lbls.cpu().numpy().flatten())
            all_probs.extend(out.cpu().numpy().flatten())
            gamma_batches.append(g.cpu().numpy())
            beta_batches.append(b.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    gamma_all  = np.concatenate(gamma_batches, axis=0)
    beta_all   = np.concatenate(beta_batches,  axis=0)

    acc = (all_preds == all_labels).mean()
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    auc_val = auc(fpr, tpr)
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()

    print('\n' + '='*60)
    print(f'FiLM MODEL EVALUATION ({tag})')
    print('='*60)
    print(f'Accuracy:    {acc:.4f} ({acc*100:.2f}%)')
    print(f'AUC:         {auc_val:.4f}')
    print(f'Sensitivity: {tp/(tp+fn):.4f}')
    print(f'Specificity: {tn/(tn+fp):.4f}')
    print(f'Gamma: mean={gamma_all.mean():.4f}, std={gamma_all.std():.4f}')
    print(f'Beta:  mean={beta_all.mean():.4f},  std={beta_all.std():.4f}')
    print(f'Confusion Matrix:')
    print(f'              Predicted')
    print(f'                     Non-responder  Responder')
    print(f'Actual  Non-responder  {cm[0,0]:4d}  {cm[0,1]:4d}')
    print(f'        Responder      {cm[1,0]:4d}  {cm[1,1]:4d}')

    return acc, auc_val, fpr, tpr

# Load best FiLM checkpoint
print(f'Loading best FiLM checkpoint: {FILM_CKPT}')
film_model.load_state_dict(torch.load(FILM_CKPT, map_location=device))

film_valid_acc, film_valid_auc, _, _ = evaluate_film_model(
    film_model, validloader, device, tag='VALID'
)
film_test_acc, film_test_auc, film_fpr, film_tpr = evaluate_film_model(
    film_model, testloader, device, tag='TEST'
)

# ── Final comparison table ────────────────────────────────────────
print('\n' + '='*60)
print('FINAL COMPARISON: Baseline vs FiLM (Frozen Backbone)')
print('='*60)
print(f'{"Metric":25s}  {"Baseline":>10s}  {"FiLM":>10s}  {"Delta":>10s}')
print('-'*60)
print(f'{"Valid Accuracy":25s}  {bl_valid_acc:>10.4f}  {film_valid_acc:>10.4f}  {film_valid_acc-bl_valid_acc:>+10.4f}')
print(f'{"Valid AUC":25s}  {bl_valid_auc:>10.4f}  {film_valid_auc:>10.4f}  {film_valid_auc-bl_valid_auc:>+10.4f}')
print(f'{"Test Accuracy":25s}  {bl_test_acc:>10.4f}  {film_test_acc:>10.4f}  {film_test_acc-bl_test_acc:>+10.4f}')
print(f'{"Test AUC":25s}  {bl_test_auc:>10.4f}  {film_test_auc:>10.4f}  {film_test_auc-bl_test_auc:>+10.4f}')

# ── ROC curve comparison ──────────────────────────────────────────
plt.figure(figsize=(8, 6))
plt.plot(bl_fpr,   bl_tpr,   linewidth=2, label=f'Baseline  AUC={bl_test_auc:.4f}')
plt.plot(film_fpr, film_tpr, linewidth=2, label=f'FiLM      AUC={film_test_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve: Baseline vs FiLM (Frozen Backbone)', fontweight='bold', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_comparison_baseline_vs_film.png', dpi=300)
plt.show()
print('\n✅ Saved: roc_comparison_baseline_vs_film.png')

# ── Per cell type FiLM analysis ───────────────────────────────────
print('\n--- FiLM gamma/beta by cell type ---')
meta_types = {
    'CD4+EM':       torch.tensor([[1.,0.,0.,0.,1.,0.]]),
    'CD4+CM':       torch.tensor([[1.,0.,0.,0.,0.,1.]]),
    'CD8+Naive':    torch.tensor([[0.,1.,1.,0.,0.,0.]]),
    'CD8+Effector': torch.tensor([[0.,1.,0.,1.,0.,0.]]),
}
film_model.eval()
with torch.no_grad():
    for lbl, mv in meta_types.items():
        g, b = film_model.get_film_params(mv.to(device))
        gn, bn = g.cpu().numpy(), b.cpu().numpy()
        print(f'  {lbl:15s}: gamma mean={gn.mean():.4f} std={gn.std():.4f} | beta mean={bn.mean():.4f} std={bn.std():.4f}')
